# Process Full Dataset - Method 3 (DICOM Windowing)

This notebook processes the ENTIRE dataset:
- **99,999 synthetic images**: Crop from (640, 512) → (512, 512)
- **163,568 original images**: Apply Method 3 (DICOM windowing from metadata)

**Output folders:**
- `/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_synthetic_resized/`
- `/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_original_method3/`

**⚠️ This will take 30-60 minutes to complete!**


In [ ]:
import os
import cv2
import numpy as np
import json
import glob
from tqdm import tqdm
import time
import random

print("Libraries imported successfully!")
print(f"Current time: {time.strftime('%Y-%m-%d %H:%M:%S')}")


Libraries imported successfully!
Current time: 2025-10-25 00:25:09


## Step 1: Setup Paths


In [2]:
# Input paths (READ ONLY - will never be modified)
SYNTHETIC_INPUT = '/raid/mpsych/OMAMA/DATA/data/train/'
NPZ_INPUT = '/hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/images/'
METADATA_INPUT = '/hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/metadata/'

# Output paths (NEW - will be created)
OUTPUT_BASE = '/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/'
SYNTHETIC_OUTPUT = os.path.join(OUTPUT_BASE, 'full_synthetic_resized')
ORIGINAL_OUTPUT = os.path.join(OUTPUT_BASE, 'full_original_method3')

# Create output directories
os.makedirs(SYNTHETIC_OUTPUT, exist_ok=True)
os.makedirs(ORIGINAL_OUTPUT, exist_ok=True)

print("="*70)
print("PATH SETUP")
print("="*70)
print("\n✅ INPUT PATHS (READ ONLY):")
print(f"  Synthetic: {SYNTHETIC_INPUT}")
print(f"  NPZ:       {NPZ_INPUT}")
print(f"  Metadata:  {METADATA_INPUT}")

print("\n✅ OUTPUT PATHS (NEW):")
print(f"  Synthetic: {SYNTHETIC_OUTPUT}")
print(f"  Original:  {ORIGINAL_OUTPUT}")

# Verify input paths exist
print("\n✅ VERIFICATION:")
print(f"  Synthetic path exists: {os.path.exists(SYNTHETIC_INPUT)}")
print(f"  NPZ path exists: {os.path.exists(NPZ_INPUT)}")
print(f"  Metadata path exists: {os.path.exists(METADATA_INPUT)}")
print("="*70)


PATH SETUP

✅ INPUT PATHS (READ ONLY):
  Synthetic: /raid/mpsych/OMAMA/DATA/data/train/
  NPZ:       /hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/images/
  Metadata:  /hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/metadata/

✅ OUTPUT PATHS (NEW):
  Synthetic: /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_synthetic_resized
  Original:  /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_original_method3

✅ VERIFICATION:
  Synthetic path exists: True
  NPZ path exists: True
  Metadata path exists: True


## Step 2: Get File Lists


In [3]:
# Get all synthetic PNG files
synthetic_files = sorted(glob.glob(os.path.join(SYNTHETIC_INPUT, '*.png')))

# Get all NPZ files
npz_files = sorted(glob.glob(os.path.join(NPZ_INPUT, '*.npz')))

print("="*70)
print("FILE INVENTORY")
print("="*70)
print(f"\n📊 Synthetic images found: {len(synthetic_files):,}")
print(f"📊 Original NPZ files found: {len(npz_files):,}")
print(f"\n📊 TOTAL images to process: {len(synthetic_files) + len(npz_files):,}")
print("="*70)


FILE INVENTORY

📊 Synthetic images found: 99,999
📊 Original NPZ files found: 163,568

📊 TOTAL images to process: 263,567


## Step 3: Process Synthetic Images (640x512 → 512x512)


In [4]:
def center_crop_640_to_512(img):
    """Center crop from (640, 512) to (512, 512)"""
    if img.shape[0] == 640:
        return img[64:576, :]
    return img

print("="*70)
print("PROCESSING SYNTHETIC IMAGES")
print("="*70)
print(f"\nStarting at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total to process: {len(synthetic_files):,}\n")

start_time = time.time()
synthetic_success = 0
synthetic_errors = []

for syn_file in tqdm(synthetic_files, desc="Cropping synthetic images"):
    try:
        # Load image
        img = cv2.imread(syn_file, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            synthetic_errors.append((os.path.basename(syn_file), "Failed to load"))
            continue
        
        # Center crop
        cropped = center_crop_640_to_512(img)
        
        # Save with same filename
        output_path = os.path.join(SYNTHETIC_OUTPUT, os.path.basename(syn_file))
        cv2.imwrite(output_path, cropped)
        
        synthetic_success += 1
        
    except Exception as e:
        synthetic_errors.append((os.path.basename(syn_file), str(e)))

elapsed = time.time() - start_time

print(f"\n✅ Successfully processed: {synthetic_success:,}/{len(synthetic_files):,}")
print(f"❌ Errors: {len(synthetic_errors):,}")
print(f"⏱️  Time elapsed: {elapsed/60:.2f} minutes")
print(f"✅ Saved to: {SYNTHETIC_OUTPUT}")

if len(synthetic_errors) > 0:
    print(f"\n⚠️  First 10 errors:")
    for filename, error in synthetic_errors[:10]:
        print(f"  {filename}: {error}")
        
print("="*70)


PROCESSING SYNTHETIC IMAGES

Starting at: 2025-10-25 00:25:11
Total to process: 99,999



Cropping synthetic images: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99999/99999 [10:36<00:00, 157.21it/s]


✅ Successfully processed: 99,999/99,999
❌ Errors: 0
⏱️  Time elapsed: 10.60 minutes
✅ Saved to: /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_synthetic_resized


## Step 4: Define Method 3 (DICOM Windowing) Function


In [5]:
def extract_value(param, default):
    """Extract numeric value from parameter that might be a list or single value"""
    if isinstance(param, list):
        return float(param[0]) if len(param) > 0 else default
    return float(param) if param is not None else default

def apply_dicom_windowing(pixel_array, window_center, window_width, 
                          rescale_intercept=0, rescale_slope=1):
    """
    Apply DICOM windowing to convert raw pixel values to display range [0, 255]
    
    This is METHOD 3 - uses WindowCenter and WindowWidth from metadata
    """
    # Convert to float for calculations
    pixel_array = pixel_array.astype(np.float32)
    
    # Apply rescale slope and intercept
    pixel_array = pixel_array * rescale_slope + rescale_intercept
    
    # Calculate window min and max
    img_min = window_center - window_width / 2
    img_max = window_center + window_width / 2
    
    # Apply windowing
    windowed = np.clip(pixel_array, img_min, img_max)
    
    # Normalize to [0, 255]
    windowed = ((windowed - img_min) / (img_max - img_min) * 255).astype(np.uint8)
    
    return windowed

print("✅ Method 3 (DICOM windowing) functions defined successfully!")


✅ Method 3 (DICOM windowing) functions defined successfully!


## Step 5: Process Original NPZ Images (Method 3)


In [6]:
print("="*70)
print("PROCESSING ORIGINAL NPZ IMAGES (METHOD 3)")
print("="*70)
print(f"\nStarting at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total to process: {len(npz_files):,}\n")

start_time = time.time()
original_success = 0
original_errors = []

for npz_file in tqdm(npz_files, desc="Processing NPZ files with Method 3"):
    try:
        # Get image ID (filename without extension)
        image_id = os.path.splitext(os.path.basename(npz_file))[0]
        
        # Load NPZ file
        npz_data = np.load(npz_file)
        
        # Get the array - use first available key
        key = npz_data.files[0]
        raw_pixels = npz_data[key]
        
        # Load corresponding metadata JSON
        json_file = os.path.join(METADATA_INPUT, f"{image_id}.json")
        
        if not os.path.exists(json_file):
            original_errors.append((image_id, "JSON metadata not found"))
            continue
            
        with open(json_file, 'r') as f:
            metadata = json.load(f)
        
        # Extract windowing parameters (handle lists!)
        window_center = extract_value(metadata.get('WindowCenter'), 2048.0)
        window_width = extract_value(metadata.get('WindowWidth'), 4096.0)
        rescale_intercept = extract_value(metadata.get('RescaleIntercept'), 0)
        rescale_slope = extract_value(metadata.get('RescaleSlope'), 1)
        
        # Apply Method 3 (DICOM windowing)
        windowed_img = apply_dicom_windowing(
            raw_pixels, 
            window_center, 
            window_width,
            rescale_intercept,
            rescale_slope
        )
        
        # Save as PNG
        output_path = os.path.join(ORIGINAL_OUTPUT, f"{image_id}.png")
        cv2.imwrite(output_path, windowed_img)
        
        original_success += 1
        
    except Exception as e:
        original_errors.append((image_id, str(e)))

elapsed = time.time() - start_time

print(f"\n✅ Successfully processed: {original_success:,}/{len(npz_files):,}")
print(f"❌ Errors: {len(original_errors):,}")
print(f"⏱️  Time elapsed: {elapsed/60:.2f} minutes")
print(f"✅ Saved to: {ORIGINAL_OUTPUT}")

if len(original_errors) > 0:
    print(f"\n⚠️  First 10 errors:")
    for img_id, error in original_errors[:10]:
        print(f"  {img_id}: {error}")
        
print("="*70)


PROCESSING ORIGINAL NPZ IMAGES (METHOD 3)

Starting at: 2025-10-25 00:35:47
Total to process: 163,568



 ... (more hidden) ...


✅ Successfully processed: 163,568/163,568
❌ Errors: 0
⏱️  Time elapsed: 21.96 minutes
✅ Saved to: /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_original_method3


## Step 6: Final Verification


## Step 7: Verify Pixel Value Range [0, 255]


In [ ]:
print("="*70)
print("VERIFYING PIXEL VALUE RANGES [0, 255]")
print("="*70)
print("\nChecking 100 random samples from each dataset...\n")

# Check synthetic
syn_check = random.sample(synthetic_output_files, min(100, len(synthetic_output_files)))
syn_min_vals = []
syn_max_vals = []
syn_out_of_range = []

for f in tqdm(syn_check, desc="Checking synthetic pixel ranges"):
    img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        syn_min_vals.append(img.min())
        syn_max_vals.append(img.max())
        if img.min() < 0 or img.max() > 255:
            syn_out_of_range.append(os.path.basename(f))

# Check original
orig_check = random.sample(original_output_files, min(100, len(original_output_files)))
orig_min_vals = []
orig_max_vals = []
orig_out_of_range = []

for f in tqdm(orig_check, desc="Checking original pixel ranges"):
    img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        orig_min_vals.append(img.min())
        orig_max_vals.append(img.max())
        if img.min() < 0 or img.max() > 255:
            orig_out_of_range.append(os.path.basename(f))

print("\n" + "="*70)
print("PIXEL RANGE VERIFICATION RESULTS")
print("="*70)

print("\n📊 Synthetic Images:")
print(f"  Global Min: {min(syn_min_vals) if syn_min_vals else 'N/A'}")
print(f"  Global Max: {max(syn_max_vals) if syn_max_vals else 'N/A'}")
print(f"  Out of range [0-255]: {len(syn_out_of_range)}/100")
if syn_out_of_range:
    print(f"  ⚠️ Problem files: {syn_out_of_range[:5]}")

print("\n📊 Original Images:")
print(f"  Global Min: {min(orig_min_vals) if orig_min_vals else 'N/A'}")
print(f"  Global Max: {max(orig_max_vals) if orig_max_vals else 'N/A'}")
print(f"  Out of range [0-255]: {len(orig_out_of_range)}/100")
if orig_out_of_range:
    print(f"  ⚠️ Problem files: {orig_out_of_range[:5]}")

if len(syn_out_of_range) == 0 and len(orig_out_of_range) == 0:
    print("\n✅ ALL IMAGES ARE PROPERLY NORMALIZED TO [0, 255]!")
else:
    print("\n⚠️ WARNING: Some images are out of range!")

print("="*70)


In [7]:
# Count output files
synthetic_output_files = glob.glob(os.path.join(SYNTHETIC_OUTPUT, '*.png'))
original_output_files = glob.glob(os.path.join(ORIGINAL_OUTPUT, '*.png'))

print("="*70)
print("FINAL VERIFICATION")
print("="*70)

print("\n📊 OUTPUT FILE COUNTS:")
print(f"  Synthetic (resized): {len(synthetic_output_files):,}")
print(f"  Original (Method 3): {len(original_output_files):,}")

print("\n📊 PROCESSING SUMMARY:")
print(f"  Synthetic success rate: {synthetic_success}/{len(synthetic_files)} ({synthetic_success/len(synthetic_files)*100:.2f}%)")
print(f"  Original success rate: {original_success}/{len(npz_files)} ({original_success/len(npz_files)*100:.2f}%)")

print("\n✅ OUTPUT LOCATIONS:")
print(f"  Synthetic: {SYNTHETIC_OUTPUT}")
print(f"  Original:  {ORIGINAL_OUTPUT}")

print("\n" + "="*70)
print("PROCESSING COMPLETE!")
print("="*70)
print(f"\nCompleted at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✨ Next step: Run Notebook 12 to verify 100 random samples!")
print("="*70)


FINAL VERIFICATION

📊 OUTPUT FILE COUNTS:
  Synthetic (resized): 99,999
  Original (Method 3): 163,568

📊 PROCESSING SUMMARY:
  Synthetic success rate: 99999/99999 (100.00%)
  Original success rate: 163568/163568 (100.00%)

✅ OUTPUT LOCATIONS:
  Synthetic: /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_synthetic_resized
  Original:  /hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_original_method3

PROCESSING COMPLETE!

Completed at: 2025-10-25 00:57:46

✨ Next step: Run Notebook 12 to verify 100 random samples!
